# Notebook 1: Breast Cancer Classification — Your First Classifiers

**BMI 503/511 — Machine Learning: Classification Methods**  
**Instructor:** Pratik Dutta, Ph.D. | Stony Brook University

---

## Learning Objectives
1. Load and explore a real clinical dataset (Wisconsin Breast Cancer)
2. Understand the train/test split paradigm
3. Train 3 classifiers: Logistic Regression, Decision Tree, Random Forest
4. Evaluate with confusion matrix and classification report
5. Visualize a decision tree

**Estimated time: ~25 minutes**

## 1. Setup & Data Loading

We use the **Wisconsin Breast Cancer** dataset — a classic clinical ML dataset.  
- 569 samples, 30 features (computed from cell nucleus images)
- **Binary classification:** Malignant (212) vs Benign (357)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = malignant, 1 = benign
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['diagnosis'].value_counts())
df.head()

### Quick EDA: What do the features look like?

In [ ]:
# Compare key features between malignant and benign
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

features_to_plot = ['mean radius', 'mean texture', 'mean concavity']
for ax, feat in zip(axes, features_to_plot):
    for label, color in [('Malignant', '#EF4444'), ('Benign', '#10B981')]:
        subset = df[df['diagnosis'] == label]
        ax.hist(subset[feat], bins=20, alpha=0.6, label=label, color=color)
    ax.set_xlabel(feat, fontsize=12)
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Feature Distributions by Diagnosis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Train/Test Split

**Why split?** We need to evaluate the model on data it has never seen.  
- 80% for training, 20% for testing
- `stratify` ensures class proportions are preserved
- `random_state` ensures reproducibility

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df[data.feature_names]
y = df['target']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use training statistics!

print("\n⚠️ Note: We fit the scaler on TRAINING data only, then transform test data.")
print("   This prevents data leakage!")

## 3. Train Three Classifiers

Let's train and compare:
1. **Logistic Regression** — linear, probabilistic
2. **Decision Tree** — non-linear, interpretable
3. **Random Forest** — ensemble of decision trees

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

# Train and evaluate each
results = {}
for name, model in models.items():
    # Use scaled data for Logistic Regression, raw for tree-based
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = {'accuracy': acc, 'predictions': y_pred}

    print(f"\n{'='*50}")
    print(f"{name} — Accuracy: {acc:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

## 4. Confusion Matrices — Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Malignant', 'Benign'],
                yticklabels=['Malignant', 'Benign'])
    ax.set_title(f"{name}\nAccuracy: {res['accuracy']:.3f}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("\n💡 Discussion: Look at the False Negatives (top-right).")
print("   In cancer screening, which type of error is more dangerous?")

## 5. Visualize a Decision Tree

One of the great advantages of decision trees: **we can see exactly how the model makes decisions.**

In [ ]:
from sklearn.tree import plot_tree

# Train a shallow tree for visualization
shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
shallow_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(shallow_tree,
          feature_names=data.feature_names,
          class_names=['Malignant', 'Benign'],
          filled=True, rounded=True, ax=ax,
          fontsize=9)
plt.title('Decision Tree (max_depth=3) — Breast Cancer Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Clinical insight: The tree uses features like 'worst perimeter' and")
print("   'worst concave points' — these correspond to cell shape irregularities")
print("   that pathologists also look for in histopathology.")

## 6. Feature Importance (Random Forest)

In [ ]:
# Get feature importances from Random Forest
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=data.feature_names)
top_features = importances.nlargest(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_features.sort_values().plot(kind='barh', color='#0D9488', ax=ax)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('Top 10 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 These features can guide biomarker discovery — which cell morphology")
print("   measurements are most predictive of malignancy?")

## ✅ Exercises for Students

1. **Change `max_depth`** of the Decision Tree to 2, 3, 10, None (unlimited). What happens to accuracy? To the tree visualization?
2. **Remove feature scaling** for Logistic Regression. Does accuracy change? Why?
3. **Change `n_estimators`** in Random Forest from 10 to 500. Plot accuracy vs. n_estimators.
4. **Clinical question:** If you were building a screening tool, would you optimize for Precision or Recall? Change the threshold and see what happens.